https://plotly.com/python/3d-volume-plots/

In [ ]:
import astropy.units as u
from spectral_cube import SpectralCube
import plotly.graph_objects as go
import numpy as np

# Figure size in pixes
WIDTH = 1000
HEIGHT = 1100

# Minimal flux values for isosurfaces
MIN_FLUX = 4.
MAX_FLUX = 30.

# Limits in Vrad 
VRAD_MIN = -60e3 * u.m / u.s
VRAD_MAX = -30e3 * u.m / u.s

cube = SpectralCube.read("RCW79_CII_20_8_0p5.fits")

# Removing noisy pixels at the RA/Dec border and get only a slice in vrad.
cube = cube[:, 15:-15, 15:-15] 

# Cut on the desired Vrad limits
cube = cube.spectral_slab(VRAD_MIN, VRAD_MAX)

# Reduce the number of pixels for the note book
cube = cube[::3, ::3, ::3]

# Arrange the cube in mesh grids
ra, dec, vrad, flux = (
    np.zeros(cube.shape, dtype=float),
    np.zeros(cube.shape, dtype=float),
    np.zeros(cube.shape, dtype=float),
    np.zeros(cube.shape, dtype=float),
)

for i in range(cube.shape[2]):
    ra[:, :, i] = cube.world[0, 0, i][2].value

for i in range(cube.shape[1]):
    dec[:, i, :] = cube.world[0, i, 0][1].value

for i in range(cube.shape[0]):
    vrad[i, :, :] = cube.world[i, 0, 0][0].value

flux = cube.unmasked_data[:]

# Swap RA and Dec axis for the plot.
ra = np.swapaxes(ra, 1, 2)
dec = np.swapaxes(dec, 1, 2)
vrad = np.swapaxes(vrad, 1, 2)
flux = np.swapaxes(flux, 1, 2)

# To swtich to log sclae in flux
#mask = (flux <= 0)
#flux[~mask] = np.log10(flux[~mask])
#flux[mask] = 0
#MIN_FLUX = np.log10(MIN_FLUX)
#MAX_FLUX = np.log10(MAX_FLUX)

fig = go.Figure(data=go.Volume(
    x=vrad.flatten(),
    y=ra.flatten(),
    z=dec.flatten(),
    value=flux.flatten(),
    isomin=MIN_FLUX,
    isomax=MAX_FLUX,
    opacity=0.05, # needs to be small to see through all surfaces
    surface_count=40, # needs to be a large number for good volume rendering
    colorscale='viridis',
    caps= dict(x_show=False, y_show=False, z_show=False), # no caps
    )
)

# RA / Dec view
zfactor = 30.
scene_radec = {
    'eye': {'x': zfactor*2.2, 'y': zfactor*0.8, 'z': zfactor*0.2},
    'center': {'x': 0, 'y': 0, 'z': -.1},
}

# RA / Vrad view
zfactor = 50.
scene_ravrad = {
    'eye': {'x': zfactor*-2, 'y': zfactor*-10, 'z': zfactor*2.4},
    'center': {'x': 0, 'y': 0, 'z': -.1},
}

fig.update_layout(
        scene = {
            'xaxis_title': 'vrad [m/s]',
            'yaxis_title': 'RA [deg]',
            'zaxis_title': 'Dec [deg]',
        },

        width=WIDTH,
        height=HEIGHT,
        margin={'r': 10, 'b': 10, 'l': 10, 't': 10},
        scene_camera=scene_ravrad,
)

fig.update_scenes(
    aspectmode='manual',
    aspectratio = {
        'x': cube.shape[0],
        'y': cube.shape[1],
        'z': cube.shape[2],
    }
)
fig.show()